# Assignment Module 2: Aircraft Classification

The goal of this assignment is to implement a neural network that classifies images of 100 aircraft model variants from the [Fine-Grained Visual Classification of Aircraft (**FGVC-Aircraft**) dataset](https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/). The assignment is divided into two parts: first, you will be asked to implement your own neural network for image classification from scratch; then, you will fine-tune a pretrained network provided by PyTorch.

![](https:///raw.githubusercontent.com/CVLAB-Unibo/ipcv-assignment-2/master/fgvc_aircraft_variants.svg)

## Part 0 — 环境与配置

环境自动检测（Colab / 本地），设置 `DATA_ROOT`、`DEVICE`、随机种子。

In [ ]:
import os, sys, math, time, json, random, collections
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.datasets import FGVCAircraft
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
DATA_ROOT = Path("/content/data") if IN_COLAB else (Path.home() / "data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()
torch.backends.cudnn.benchmark = True

print(f"torch {torch.__version__} | tv {torchvision.__version__}")
print(f"Colab={IN_COLAB} | device={DEVICE} | data_root={DATA_ROOT}")

## Dataset

Download and acces the dataset through its official [PyTorch `FGVCAircraft` class](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FGVCAircraft.html) (by setting its constructor argument `annotation_level` to `'variant'`).

### 数据集说明

FGVC-Aircraft，100 个 *variant* 类，train/val/test 各 ~3333 张，类别近似均衡。
每张图底部有约 20px 版权条，resize 前先裁掉。
`train` 用于训练、`val` 用于选模型/早停/调参、`test` 只用于报告最终数字。
首次下载约 2.75GB（牛津 VGG 官网，若失败需手动下 tar 包并放到 `DATA_ROOT`）。

In [ ]:
NUM_CLASSES = 100

def load_raw(split):
    return FGVCAircraft(root=str(DATA_ROOT), split=split,
                        annotation_level="variant", download=True, transform=None)

raw_train, raw_val, raw_test = load_raw("train"), load_raw("val"), load_raw("test")
print(f"train {len(raw_train)} | val {len(raw_val)} | test {len(raw_test)}")
print(f"#classes = {len(raw_train.classes)}  e.g. {raw_train.classes[:4]}")
assert len(raw_train.classes) == NUM_CLASSES

In [ ]:
class CropBottomBanner:
    """去掉 FGVC-Aircraft 底部 ~20px 版权条"""
    def __init__(self, px=20): self.px = px
    def __call__(self, img):
        w, h = img.size
        return img.crop((0, 0, w, h - self.px))

IMG_SIZE  = 128
RESIZE_TO = int(round(IMG_SIZE * 1.14))   # 先放大再 center-crop，保留长宽比

def compute_mean_std(base, n=1500):
    tf = transforms.Compose([CropBottomBanner(20),
                             transforms.Resize(RESIZE_TO),
                             transforms.CenterCrop(IMG_SIZE),
                             transforms.ToTensor()])
    idx = np.random.default_rng(SEED).choice(len(base), min(n, len(base)), replace=False)
    mean = torch.zeros(3); sq = torch.zeros(3)
    for i in idx:
        x = tf(base[i][0]); mean += x.mean((1, 2)); sq += (x**2).mean((1, 2))
    mean /= len(idx); std = (sq/len(idx) - mean**2).sqrt()
    return mean.tolist(), std.tolist()

FGVC_MEAN, FGVC_STD = compute_mean_std(raw_train)
print("mean", [round(v, 4) for v in FGVC_MEAN], "std", [round(v, 4) for v in FGVC_STD])

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_train_tf(img_size=IMG_SIZE, mean=None, std=None,
                   augment=True, rrc=True, hflip=True, jitter=True):
    mean = mean or FGVC_MEAN; std = std or FGVC_STD
    steps = [CropBottomBanner(20)]
    if augment and rrc:
        steps.append(transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0), ratio=(0.8, 1.25)))
    else:
        steps += [transforms.Resize(int(round(img_size*1.14))), transforms.CenterCrop(img_size)]
    if augment and hflip:  steps.append(transforms.RandomHorizontalFlip())
    if augment and jitter: steps.append(transforms.ColorJitter(0.2, 0.2, 0.2))
    steps += [transforms.ToTensor(), transforms.Normalize(mean, std)]
    return transforms.Compose(steps)

def build_eval_tf(img_size=IMG_SIZE, mean=None, std=None):
    mean = mean or FGVC_MEAN; std = std or FGVC_STD
    return transforms.Compose([CropBottomBanner(20),
                               transforms.Resize(int(round(img_size*1.14))),
                               transforms.CenterCrop(img_size),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])

In [ ]:
class Transformed(Dataset):
    def __init__(self, base, tf): self.base, self.tf = base, tf
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        img, y = self.base[i]; return self.tf(img), y

def make_loaders(img_size=IMG_SIZE, batch_size=128, mean=None, std=None,
                 augment=True, rrc=True, hflip=True, jitter=True, num_workers=2):
    ttf = build_train_tf(img_size, mean, std, augment, rrc, hflip, jitter)
    etf = build_eval_tf(img_size, mean, std)
    common = dict(num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"))
    return (DataLoader(Transformed(raw_train, ttf), batch_size, shuffle=True,  drop_last=True, **common),
            DataLoader(Transformed(raw_val,   etf), batch_size, shuffle=False,                 **common),
            DataLoader(Transformed(raw_test,  etf), batch_size, shuffle=False,                 **common))

## EDA

In [ ]:
def denorm(x, mean=None, std=None):
    mean = torch.tensor(mean or FGVC_MEAN).view(3, 1, 1)
    std  = torch.tensor(std  or FGVC_STD ).view(3, 1, 1)
    return (x*std + mean).clamp(0, 1)

# 1) 增强后样本网格
tl, _, _ = make_loaders(batch_size=16, num_workers=0)
xb, yb = next(iter(tl))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, im, y in zip(axes.flat, xb, yb):
    ax.imshow(denorm(im).permute(1, 2, 0).numpy()); ax.axis("off")
    ax.set_title(raw_train.classes[y][:14], fontsize=7)
plt.suptitle("augmented training samples"); plt.tight_layout(); plt.show()

# 2) 类别分布
_lbl = getattr(raw_train, "_labels", None)
lbl = np.array(_lbl if _lbl is not None else [y for _, y in raw_train])
cnt = np.bincount(lbl, minlength=NUM_CLASSES)
plt.figure(figsize=(10, 2.5)); plt.bar(range(NUM_CLASSES), cnt)
plt.title(f"train imgs/class  (min {cnt.min()}, max {cnt.max()})"); plt.show()

# 3) 原图尺寸
s = np.array([raw_train[i][0].size for i in np.random.default_rng(0).integers(0, len(raw_train), 200)])
print(f"raw W {s[:,0].min()}-{s[:,0].max()} | raw H {s[:,1].min()}-{s[:,1].max()}")

## Part 1: design your own network

Your goal is to implement a convolutional neural network for image classification and train it from scratch on `FGVCAircraft`. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~50%. You are free to achieve this however you want, except for a few rules you must follow:

- Compile this notebook by displaying the results obtained by the best model you found throughout your experimentation; then show how, by removing some of its components, its performance drops. In other words, do an *ablation study* to prove that your design choices have a positive impact on the final result.

- Do not instantiate an off-the-self PyTorch network. Instead, construct your network as a composition of existing PyTorch layers. In more concrete terms, you can use e.g. `torch.nn.Linear`, but you cannot use e.g. `torchvision.models.alexnet`.

- Show your results and ablations with plots, tables, images, etc. — the clearer, the better.

Don't be too concerned with your model performance: the ~50% is just to give you an idea of when to stop. Keep in mind that a thoroughly justified model with lower accuracy will be rewarded more points than a poorly experimentally validated model with higher accuracy.

## Part 2: fine-tune an existing network

Your goal is to fine-tune a pretrained ResNet-18 model on `FGVCAircraft`. Use the implementation provided by PyTorch, i.e. the opposite of part 1. Specifically, use the PyTorch ResNet-18 model pretrained on ImageNet-1K (V1). Divide your fine-tuning into two parts:

2A. First, fine-tune the ResNet-18 with the same training hyperparameters you used for your best model in part 1.

2B. Then, tweak the training hyperparameters to increase the accuracy on the test split. Justify your choices by analyzing the training plots and/or citing sources that guided you in your decisions — papers, blog posts, YouTube videos, or whatever else you may find useful. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~70%.